# 10 — Harness validation (week 1 gate)

Before any arm is trusted, the harness must show three things:

1. **Effectiveness is right.** BM25 (k1=0.9, b=0.4) on MS MARCO passage should reproduce the
   Pyserini reference on TREC DL 19/20.
2. **Latency is measured sanely.** Single-stream BM25 latency is stable across repetitions, and
   rankings are deterministic.
3. **The load driver behaves as queueing theory says.** With `workers` fixed, latency rises once
   offered concurrency exceeds the worker count, and throughput plateaus.

Run on the measurement machine with the governor pinned:
`sudo cpupower frequency-set -g performance`.

In [1]:
import json
import pandas as pd
from cs4406_sem_project import data
from cs4406_sem_project.bench import sysinfo
from cs4406_sem_project.config import LoadConfig, PipelineConfig, ServingConfig
from cs4406_sem_project.paths import index_path

ALLOW_UNPINNED = False  # set True only for a dry run; results are then not reportable
sysinfo.check_governor(ALLOW_UNPINNED)
print(json.dumps(sysinfo.snapshot(), indent=2))

{
  "hostname": "imanpc",
  "cpu_model": "13th Gen Intel(R) Core(TM) i5-13500",
  "logical_cpus": 20,
  "physical_cpus": 14,
  "governors": [
    "performance"
  ],
  "mem_total_bytes": 16412991488,
  "mem_available_bytes": 12376023040,
  "loadavg_1m": 0.822265625,
  "python": "3.12.14",
  "kernel": "7.2.6-arch2-1",
  "thread_env": {
    "OMP_NUM_THREADS": null,
    "OPENBLAS_NUM_THREADS": null,
    "MKL_NUM_THREADS": null,
    "NUMBA_NUM_THREADS": null,
    "RAYON_NUM_THREADS": null
  },
  "packages": {
    "bm25s": "0.3.13",
    "numpy": "2.5.3",
    "ir_datasets": "0.6.3",
    "faiss-cpu": "1.15.1",
    "pyseismic-lsr": "0.5.2"
  }
}


## 1. Build the BM25 index (one-off; the first run also downloads MS MARCO via ir_datasets)

In [2]:
from cs4406_sem_project.first_stage import bm25

if not (index_path("bm25") / "build_meta.json").exists():
    bm25.build()
meta = json.loads((index_path("bm25") / "build_meta.json").read_text())
size_gb = sum(p.stat().st_size for p in index_path("bm25").iterdir()) / 1e9
print(f"docs indexed: {meta['num_docs']:,}")
print(f"build time:   {meta['build_seconds'] / 60:.1f} min (tokenize {meta['tokenize_seconds'] / 60:.1f} min)")
print(f"index size:   {size_gb:.2f} GB on disk")

docs indexed: 8,841,823
build time:   4.6 min (tokenize 4.1 min)
index size:   2.43 GB on disk


## 2. Effectiveness vs the Pyserini reference

Reference: Pyserini BM25 *default* (k1=0.9, b=0.4) MS MARCO passage runs, from the Pyserini
two-click reproduction (2CR) page for MS MARCO V1 passage. Recall@1000 uses relevance >= 2.
**Check these four numbers against the 2CR page before citing them.**

bm25s tokenizes with a regex word splitter and the Snowball English stemmer, whereas Lucene
uses its StandardAnalyzer with Porter stemming, so small deviations are expected. Tolerance:
±0.02.

In [3]:
from cs4406_sem_project.eval import evaluate_config
from cs4406_sem_project.pipeline import Pipeline

REFERENCE = {  # Pyserini 2CR, BM25 default (k1=0.9, b=0.4)
    "dl19": {"nDCG@10": 0.5058, "R(rel=2)@1000": 0.7501},
    "dl20": {"nDCG@10": 0.4796, "R(rel=2)@1000": 0.7863},
}
TOLERANCE = 0.02

a2 = PipelineConfig("A2", "bm25", 1000)
pipe = Pipeline(a2)
rows, per_query = [], {}
for s in ("dl19", "dl20"):
    pq, _ = evaluate_config(a2, s, pipeline=pipe)
    per_query[s] = pq
    for m, ref in REFERENCE[s].items():
        ours = pq[m].mean()
        rows.append({"set": s, "measure": m, "ours": round(ours, 4), "pyserini": ref,
                     "diff": round(ours - ref, 4), "ok": abs(ours - ref) <= TOLERANCE, "queries": len(pq)})
eff = pd.DataFrame(rows)
eff

/mnt/data/cs4406-sem-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,set,measure,ours,pyserini,diff,ok,queries
0,dl19,nDCG@10,0.5105,0.5058,0.0047,True,43
1,dl19,R(rel=2)@1000,0.7507,0.7501,0.0006,True,43
2,dl20,nDCG@10,0.4828,0.4796,0.0032,True,54
3,dl20,R(rel=2)@1000,0.7889,0.7863,0.0026,True,54


## 3. Single-stream latency, stability and determinism

A2 at k=100 and k=1000, concurrency 1, one worker with one thread, 3 repetitions: 500 warm-up
queries followed by up to 2,000 measured MS MARCO dev queries each.
Pass criteria: the per-repetition p95 spread is under 10% of the median p95, and the
top-10 hash of every query is identical across repetitions.

In [4]:
from cs4406_sem_project.bench.load import run_load
from cs4406_sem_project.eval import latency_summary

pool = data.queries(data.LOAD_QUERIES)
single = ServingConfig(workers=1, threads_per_worker=1, backend="process")
lat_rows, hashes = [], {}
for k in (100, 1000):
    cfg = PipelineConfig("A2", "bm25", k)
    for rep in range(3):
        df, m = run_load(cfg, LoadConfig(1, seed=0, serving=single), pool, allow_unpinned=ALLOW_UNPINNED)
        hashes[(k, rep)] = df.set_index("qid")["top10_hash"]
        lat_rows.append({"k": k, "rep": rep, **latency_summary(df["latency_ns"]),
                         "service_p50_ms": df["service_ns"].median() / 1e6, "qps": m["achieved_qps"],
                         "pss_gb": m["workers_pss_bytes"] / 1e9})
lat = pd.DataFrame(lat_rows)
lat

,k,rep,p50_ms,p95_ms,p99_ms,n,service_p50_ms,qps,pss_gb
0,100,0,12.827571,38.592184,57.289168,2000,12.230393,62.517928,2.406447
1,100,1,12.735408,39.611794,57.330590,2000,12.139580,62.651612,2.406515
2,100,2,12.518664,40.361023,59.110318,2000,11.996472,62.301836,2.384822
3,1000,0,13.071951,39.117118,57.118153,2000,12.523874,61.280347,2.406420
4,1000,1,13.367538,40.040253,57.929413,2000,12.806523,60.374859,2.405394
5,1000,2,13.259423,39.104520,57.678585,2000,12.658318,61.013939,2.405298


In [5]:
stability = lat.groupby("k")["p95_ms"].agg(["median", "min", "max"])
stability["spread_pct"] = 100 * (stability["max"] - stability["min"]) / stability["median"]
stability["ok"] = stability["spread_pct"] < 10
deterministic = {k: all(hashes[(k, 0)].equals(hashes[(k, r)]) for r in (1, 2)) for k in (100, 1000)}
print("deterministic rankings across repetitions:", deterministic)
stability

deterministic rankings across repetitions: {100: True, 1000: True}


,median,min,max,spread_pct,ok
k,,,,,
100,39.611794,38.592184,40.361023,4.465434,True
1000,39.117118,39.104520,40.040253,2.392134,True


## 4. Load-driver sanity: fixed capacity, rising concurrency

4 workers, A2 at k=1000, concurrency 1, 2, 4, 8 and 16, one short repetition each. Expected:
throughput grows roughly linearly up to 4 streams and then flattens. p50 latency stays near the
service time up to 4 streams; above that, queueing (`queue_ns`) accounts for the extra latency.

In [6]:
four = ServingConfig(workers=4, threads_per_worker=1, backend="process")
cfg = PipelineConfig("A2", "bm25", 1000)
sweep = []
for c in (1, 2, 4, 8, 16):
    df, m = run_load(cfg, LoadConfig(c, warmup_queries=200, max_queries=1000, serving=four), pool, allow_unpinned=ALLOW_UNPINNED)
    sweep.append({"concurrency": c, **latency_summary(df["latency_ns"]),
                  "queue_p50_ms": df["queue_ns"].median() / 1e6, "service_p50_ms": df["service_ns"].median() / 1e6,
                  "qps": m["achieved_qps"]})
pd.DataFrame(sweep)

,concurrency,p50_ms,p95_ms,p99_ms,n,queue_p50_ms,service_p50_ms,qps
0,1,13.827637,40.019232,56.819717,1000,0.287890,13.214458,59.102431
1,2,17.898544,50.448312,74.776691,1000,0.304194,17.232940,92.151127
2,4,25.033390,75.719459,108.777489,1000,0.307020,24.391701,128.557963
3,8,55.352355,111.274159,138.594659,1000,28.738975,23.678425,132.183563
4,16,116.936300,177.593404,204.690819,1000,88.767162,23.942258,131.954850


## 5. Gate verdict

In [7]:
print("effectiveness within tolerance:", bool(eff["ok"].all()))
print("latency stable (p95 spread < 10%):", bool(stability["ok"].all()))
print("rankings deterministic:", all(deterministic.values()))

effectiveness within tolerance: True
latency stable (p95 spread < 10%): True
rankings deterministic: True
